# 04. Dataset Labeling using the Teacher Model

In this notebook, we:
1) Load the dataset of $(s,u) \in \mathcal{S} \times \mathcal{U}$ pairs.
2) Use the teacher to labeling each $(s,u)$ pair with a sequence of actions $a \in \mathcal{A}^*$.
3) Save the resulting dataset of $(s,u,a)$ triplets.

As we are using `GPT-5.1`, we use **OpenAI APIs** via the `openai` library.

In [3]:
import dotenv

dotenv.load_dotenv()

True

In [4]:
from src.core.types import *
from src.core.clients.openai_client import OpenAIClient, ProcessingMode
from src.doom.utils.doom_game_state import DoomGameState
from dataclasses import asdict
from pathlib import Path
from openai.types.responses import Response as OpenAIResponse

import pandas as pd

In [5]:
%load_ext autoreload
%autoreload 2

## 1) Loading the Dataset

In [6]:
inputs = LLMCommandingInput.load_inputs(
    path=Path("data/inputs/inputs.json"),
    gstype=DoomGameState
)

inputs_lookup = {inp.id: inp for inp in inputs}

In [7]:
selected_inputs = [
    inp
    for inp in inputs
    if inp.selected_for_labelling
]

print(f"Selected inputs: {len(selected_inputs)}/{len(inputs)}")

Selected inputs: 50/2872


## 2) Preparing Teacher and System Prompt

In [6]:
teacher_client = OpenAIClient[LLMCommandingInput, LLMCommandingOutput](
    model="gpt-5.1",
    mode=ProcessingMode.BATCH, # Should be SEQUENTIAL to measure latency
    max_output_tokens=800,
    # temperature=0.0,
    reasoning_effort='low',
    working_dir=Path('data/openai'),
)

In [8]:
# Prepare Prompt
base_prompt = Path("prompts/command-execution-template-level-3.md").read_text(encoding="utf-8")
doom_root = Path("prompts/ultimate_doom")

full_prompt = base_prompt
for md_file in doom_root.rglob("*.md"):
    rel = md_file.relative_to(doom_root).with_suffix("")
    tag = "_".join(part.upper() for part in rel.parts)
    value = md_file.read_text(encoding="utf-8")
    full_prompt = full_prompt.replace(f"<{tag}>", value)

# No tool call needed at level 3 - we use DSL

In [9]:
def format_input(inp: LLMCommandingInput) -> str:
    game_state = inp.game_state.state.to_prompt_ready()
    command = inp.user_command.command.command
    return f"Game State:\n{game_state}\n\nUser Command:\n{command}"


def parse_output(response: OpenAIResponse, input_id: str, latency: float) -> LLMCommandingOutput:
    return LLMCommandingOutput(
        input_id=input_id,
        actions=response.output_text,
        reason=None,
        latency=latency,
    )


def get_id(gse: LLMCommandingInput, idx: int) -> str:
    return gse.id

In [13]:
print(full_prompt)
print(format_input(inputs[123]))

# Your Role

You are a gaming assistant for the game The Ultimate Doom. Your goal is to provide direct assistance to a human player by responding to their commands with properly chosen game actions, in the **ACTION** Domain Specific Language (DSL).

TASK INPUTS:
- The current game state.
- A user command, expressed by the player you are assisting.

TASK OUTPUT:
- A syntactically correct program written in the ACTION DSL, translating the user command into instructions for the game.

# General Instructions

- Use the provided game state only if it is relevant to the user command.
- If the command cannot be fulfilled using the available actions, output a single FAIL instruction with a brief reason.
- Otherwise, output a valid ACTION DSL program that fulfills the user command.
- Use the minimum number of actions and the shortest durations necessary.
- Output ONLY valid ACTION DSL. Do not output explanations or natural language.


# The Game The Ultimate Doom

The Ultimate Doom is a first-p

In [12]:
selected_inputs = selected_inputs[800:]

## 3) Producing Game Action Sequences

In [13]:
outputs = teacher_client.process(
    dataset=selected_inputs,
    system_prompt=full_prompt,
    tools = [], # No tools at level 3
    format_input=format_input,
    parse_output=parse_output,
    get_id=get_id,
    batch_size=200,
)

Processing mode: Batch
Model: gpt-5.1
Processing 2022 items in 11 batch(es)

Batch 1/11
Items 0 to 199 (200 total)
Status: validating | Progress: 0/0 | Failed: 0
Status: in_progress | Progress: 43/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: in_progress | Progress: 199/200 | Failed: 0
Status: completed | Progress: 200/200 | Failed: 0
Batch 1/1

In [19]:
len(outputs)
outputs[0]

LLMCommandingOutput(input_id='state-233-p0-uc0', actions='SPRINT 0.0 330.86\nINTERACT', reason=None, latency=0.0)

## 4) Saving the Result

In [20]:
# Clustering was already made earlier, so inputs are already partitioned.
# Now, considering this is just a test of the Teacher's quality (to save time):
# - Having retrieved the LLMCommandingOutputs, I can just prepare the csv
# - This time, every row in the csv should be set to be evaluated.
# For the full run: check if selected. (MAKE SURE TO CHANGE FILE NAME SO THAT I DO NOT HAVE TO REVALUATE IF THEY ALREADY CORRECT)

rows = []
for idx, output in enumerate(outputs):
    inp = inputs_lookup[output.input_id]

    row = LLMCommandingLabelledDataPoint(
        input_id=inp.id,
        game_state=inp.game_state.state.to_prompt_ready(),
        command=inp.user_command.command.command,
        command_intent=inp.user_command.command.intent,
        command_explicitness=inp.user_command.command.explicitness,
        command_atomicity=float(inp.user_command.command.atomicity),
        command_contextuality=float(inp.user_command.command.contextuality),
        game_actions=output.actions.__str__(),
        latency=output.latency,
        reason_if_failed=output.reason,
        cluster_id=inp.cluster_id,
        selected_for_labelling=inp.selected_for_labelling,
    )

    rows.append(asdict(row))

df = pd.DataFrame(rows)

In [21]:
output_path = Path("data/outputs/non-selected-data-low-reasoning-gpt5.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)